In [1]:
import json
import os
from tqdm.notebook import tqdm

In [9]:
INPUT_JSON_PATH = "output_flash.json"

OUTPUT_DIR = "data/annotations3"
MODEL_NAME_TO_ADD = "gemini"

In [ ]:
def transform_annotation(old_ann: dict) -> dict:
    species_probs = []
    if "species" in old_ann and "species_confidence" in old_ann:
        for species, prob in zip(old_ann["species"], old_ann["species_confidence"]):
            species_probs.append({"species": species, "probability": prob})

    diseases_probs = []
    if "diseases" in old_ann and "diseases_confidence" in old_ann:
        for disease, prob in zip(old_ann["diseases"], old_ann["diseases_confidence"]):
            diseases_probs.append({"disease": disease, "probability": prob})
    elif not diseases_probs:
         diseases_probs.append({"disease": "здоров", "probability": 1.0})
         
    new_ann = {
        "bbox": old_ann.get("bbox"),
        "class": old_ann.get("class_name"),
        "species_probabilities": species_probs,
        "hollows": old_ann.get("hollows"),
        "cracks": old_ann.get("cracks"),
        "stem_damage": old_ann.get("stem_damage"),
        "crown_damage": old_ann.get("crown_damage"),
        "fruiting_bodies": old_ann.get("fruiting_bodies"),
        "diseases": diseases_probs,
        "dry_branches_percentage": old_ann.get("dry_branches_percentage")
    }
    
    return new_ann

In [10]:
with open(INPUT_JSON_PATH, 'r', encoding='utf-8') as f:
    input_data = json.load(f)
    
print(f"Найдено {len(input_data)} записей в исходном файле. Начинаем преобразование...")

Найдено 210 записей в исходном файле. Начинаем преобразование...


In [11]:
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"Создана директория: {OUTPUT_DIR}")

Создана директория: data/annotations3


In [12]:
processed_count = 0

for item in tqdm(input_data, desc="Обработка записей"):
    filename = item.get("filename")
    results = item.get("results")
    
    if not filename or not results:
        tqdm.write(f"ПРЕДУПРЕЖДЕНИЕ: Пропущена запись из-за отсутствия 'filename' или 'results'.")
        continue
        
    new_annotations = [transform_annotation(res) for res in results]
    
    output_data = {
        "img_path": filename,
        "annotations": new_annotations,
        "model_name": MODEL_NAME_TO_ADD
    }
    
    base_filename = os.path.splitext(filename)[0]
    output_path = os.path.join(OUTPUT_DIR, f"{base_filename}.json")
    
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(output_data, f, ensure_ascii=False, indent=2)
        processed_count += 1
    except Exception as e:
        tqdm.write(f"ОШИБКА при сохранении файла {output_path}: {e}")

print(f"\nПреобразование завершено. Создано {processed_count} файлов в папке '{OUTPUT_DIR}'.")

Обработка записей:   0%|          | 0/210 [00:00<?, ?it/s]


Преобразование завершено. Создано 210 файлов в папке 'data/annotations3'.


Настройка и импорты

In [ ]:
!pip install open_clip_torch -q

In [ ]:
# !pip show transformers

In [1]:
import os
import json
import random
from typing import List, Dict, Any, Tuple

import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont

import matplotlib.pyplot as plt
import matplotlib.patches as patches

from huggingface_hub import login
from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection,
    Sam2Model
)
# import open_clip

# from google.colab import userdata
from tqdm import tqdm

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token, add_to_git_credential=False)

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_PATH = "tree.jpg"

GROUNDING_DINO_ID = "IDEA-Research/grounding-dino-tiny"
SAM2_ID = "facebook/sam2-hiera-base-plus"
BIOCLIP_ID = "hf-hub:imageomics/bioclip"

In [ ]:
def load_all_models() -> Dict[str, Any]:
    models = {}
    print("1/3: Grounding DINO...")
    models['dino_processor'] = AutoProcessor.from_pretrained(GROUNDING_DINO_ID)
    models['dino_model'] = AutoModelForZeroShotObjectDetection.from_pretrained(GROUNDING_DINO_ID).to(DEVICE)
    models['dino_model'].eval()

    print("2/3: SAM2...")
    models['sam_processor'] = AutoProcessor.from_pretrained(SAM2_ID)
    models['sam_model'] = Sam2Model.from_pretrained(SAM2_ID).to(DEVICE)
    models['sam_model'].eval()

    print("3/3: BioCLIP...")
    model, _, preprocess = open_clip.create_model_and_transforms(BIOCLIP_ID)
    models['bioclip_model'] = model.to(DEVICE)
    models['bioclip_preprocess'] = preprocess
    models['bioclip_tokenizer'] = open_clip.get_tokenizer(BIOCLIP_ID)
    models['bioclip_model'].eval()

    return models

In [ ]:
models = load_all_models()

In [ ]:
species_candidates = [
    # A. Основные лиственные деревья (Deciduous)
    ("берёза повислая", "Betula pendula"),
    ("берёза пушистая", "Betula pubescens"),
    ("дуб черешчатый", "Quercus robur"),
    ("дуб красный", "Quercus rubra"),
    ("клён платановидный", "Acer platanoides"),
    ("клён-явор", "Acer pseudoplatanus"),
    ("клён ясенелистный", "Acer negundo"),
    ("ясень обыкновенный", "Fraxinus excelsior"),
    ("граб", "Carpinus betulus"),
    ("вяз шершавый", "Ulmus glabra"),
    ("вяз гладкий", "Ulmus laevis"),
    ("ольха серая", "Alnus incana"),
    ("ольха черная", "Alnus glutinosa"),
    ("осина", "Populus tremula"),
    ("тополь белый", "Populus alba"),
    ("черёмуха обыкновенная", "Prunus padus"),
    ("черешня", "Prunus avium"),
    ("слива домашняя", "Prunus domestica"),
    ("яблоня домашняя", "Malus domestica"),
    ("груша обыкновенная", "Pyrus communis"),
    ("бузина чёрная", "Sambucus nigra"),
    ("рябина обыкновенная", "Sorbus aucuparia"),
    ("каштан конский", "Aesculus hippocastanum"),
    ("липа мелколистная", "Tilia cordata"),
    ("липа крупнолистная", "Tilia platyphyllos"),
    ("бук европейский", "Fagus sylvatica"),
    ("лещина обыкновенная", "Corylus avellana"),
    ("ива плакучая", "Salix babylonica"),

    # B. Хвойные деревья (Conifers)
    ("сосна обыкновенная", "Pinus sylvestris"),
    ("ель европейская", "Picea abies"),
    ("ель колючая", "Picea pungens"),
    ("лиственница европейская", "Larix decidua"),
    ("туя западная", "Thuja occidentalis"),
    ("можжевельник обыкновенный", "Juniperus communis"),
    ("пихта белая", "Abies alba"),

    # C. Декоративные/городские кустарники (Shrubs)
    ("сирень обыкновенная", "Syringa vulgaris"),
    ("форзиция", "Forsythia"),
    ("спирея", "Spiraea"),
    ("калина обыкновенная", "Viburnum opulus"),
    ("жимолость", "Lonicera"),
    ("бирючина обыкновенная", "Ligustrum vulgare"),
    ("бересклет европейский", "Euonymus europaeus"),
    ("барбарис Тунберга", "Berberis thunbergii"),
    ("шиповник собачий", "Rosa canina"),
    ("кизильник", "Cotoneaster"),
    ("снежноягодник", "Symphoricarpos"),
    ("крушина ломкая", "Frangula alnus"),
    ("пузыреплодник калинолистный", "Physocarpus opulifolius"),
    ("дёрен белый", "Cornus alba"),

    # D. Плодовые, садовые
    ("яблоня садовая", "Malus domestica"),
    ("груша садовая", "Pyrus communis"),
    ("вишня обыкновенная", "Prunus cerasus"),
    ("слива", "Prunus domestica"),
    ("облепиха крушиновидная", "Hippophae rhamnoides"),
    ("смородина/крыжовник", "Ribes"),
    ("айва обыкновенная", "Cydonia oblonga"),

    # E. Generic / полезные классы
    ("лиственное дерево", "a deciduous tree"),
    ("хвойное дерево", "a coniferous tree"),
    ("кустарник", "a shrub"),
    ("плодовое дерево", "a fruit tree"),
    ("саженец", "a sapling"),
    ("пень", "a tree stump"),
    ("мёртвое дерево", "a dead tree")
]

SYMPTOM_MAP = {
    "stem_rot": [], "hollows": [], "cracks": [], "stem_damage": [],
    "crown_damage": [], "fruiting_bodies": [], "diseases": []
}

In [ ]:
def detect_objects(
    dino_model,
    dino_processor,
    image: Image.Image
) -> Dict[str, Any]:
    text_labels = [["a tree", "a shrub"]]
    inputs = dino_processor(images=image, text=text_labels, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = dino_model(**inputs)

    results = dino_processor.post_process_grounded_object_detection(
        outputs,
        threshold=0.4,
        text_threshold=0.3,
        target_sizes=[image.size[::-1]]
    )
    return results[0]

In [ ]:
def detect_multiple_objects(
    dino_model,
    dino_processor,
    image: Image.Image,
    target_classes: List[str] = ["tree", "shrub"],
    max_objects_per_class: int = 5
) -> Dict[str, Any]:
    all_boxes = []
    all_scores = []
    all_labels = []

    for class_name in target_classes:
        text_labels = [[f"a {class_name}"]]

        inputs = dino_processor(images=image, text=text_labels, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = dino_model(**inputs)

        results = dino_processor.post_process_grounded_object_detection(
            outputs,
            threshold=0.40,
            text_threshold=0.25,
            target_sizes=[image.size[::-1]]
        )[0]

        num_found = len(results['boxes'])
        if num_found > 0:
            num_to_keep = min(num_found, max_objects_per_class)

            all_boxes.append(results['boxes'][:num_to_keep])
            all_scores.append(results['scores'][:num_to_keep])
            all_labels.extend([class_name] * num_to_keep)

    if not all_boxes:
        return {
            "boxes": torch.empty((0, 4), device=DEVICE),
            "scores": torch.empty(0, device=DEVICE),
            "text_labels": []
        }

    final_boxes = torch.cat(all_boxes, dim=0)
    final_scores = torch.cat(all_scores, dim=0)

    merged_results = {
        "scores": final_scores,
        "boxes": final_boxes,
        "text_labels": all_labels
    }

    return merged_results

In [ ]:
def segment_objects(
    sam_model,
    sam_processor,
    image: Image.Image,
    boxes: List
) -> List[np.ndarray]:
    if not boxes:
        return []
    inputs = sam_processor(image, input_boxes=[boxes], return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = sam_model(**inputs)

    multi_masks = sam_processor.image_processor.post_process_masks(
        outputs.pred_masks,
        original_sizes=[image.size[::-1]],
        target_sizes=[image.size[::-1]]
    )[0]

    final_masks = []
    for mask_set in multi_masks:
        best_mask = mask_set[0].cpu().numpy()
        final_masks.append(best_mask)

    return final_masks

In [ ]:
def build_species_prompts(latin_name: str) -> List[str]:
    return [
        f"a photo of a {latin_name}",
        f"a {latin_name} tree",
        f"leaves of a {latin_name}",
        f"bark of a {latin_name}",
        f"a branch of {latin_name}",
    ]

In [ ]:
def analyze_objects_with_bioclip(
    models: Dict, image: Image.Image, detection_result: Dict, masks: List,
    species_list: List[Tuple[str, str]]
) -> List[Dict[str, Any]]:
    final_annotations = []
    boxes = detection_result['boxes'].tolist()
    if not boxes:
        return final_annotations

    all_prompts = []
    prompt_map = []
    for i, (common_name, latin_name) in enumerate(species_list):
        prompts = build_species_prompts(latin_name)
        all_prompts.extend(prompts)
        prompt_map.extend([(i, common_name, latin_name)] * len(prompts))

    text_tokens = models['bioclip_tokenizer'](all_prompts).to(DEVICE)
    with torch.no_grad():
        text_features = models['bioclip_model'].encode_text(text_tokens)
        text_features /= text_features.norm(dim=-1, keepdim=True)

    for i, (box, mask) in enumerate(zip(boxes, masks)):
        if not np.any(mask):
            continue

        crop = image.crop(tuple(map(int, box)))
        image_input = models['bioclip_preprocess'](crop).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            image_features = models['bioclip_model'].encode_image(image_input)
            image_features /= image_features.norm(dim=-1, keepdim=True)

        all_logits = (100.0 * image_features @ text_features.T).squeeze(0)

        species_logits = torch.full((len(species_list),), -100.0, device=DEVICE)
        for prompt_idx, logit in enumerate(all_logits):
            species_idx, _, _ = prompt_map[prompt_idx]
            species_logits[species_idx] = torch.max(species_logits[species_idx], logit)

        species_probs = species_logits.softmax(dim=-1)
        top_probs, top_indices = torch.topk(species_probs, k=3)

        species_results = []
        for prob, idx in zip(top_probs.tolist(), top_indices.tolist()):
            species_results.append({
                "species": species_list[idx][0],
                "probability": round(prob, 4)
            })

        # has_hollow = find_hollows(
        #     models['dino_model'], models['dino_processor'], image, mask
        # )

        has_hollow = None

        annotation = {
            "bbox": [int(coord) for coord in box],
            "class": detection_result["text_labels"][i],
            "species_probabilities": species_results,
            "hollows": has_hollow,
            "cracks": None,
            "stem_damage": None,
            "crown_damage": None,
            "fruiting_bodies": None,
            "diseases": None,
            "dry_branches_percentage": None
        }
        final_annotations.append(annotation)

    return final_annotations

In [ ]:
result = detect_objects(models['dino_model'], models['dino_processor'], image)
boxes_raw = result['boxes'].tolist()
print(f"{len(boxes_raw)} objects")

In [ ]:
multiple_results = detect_multiple_objects(
    models['dino_model'],
    models['dino_processor'],
    image,
)
boxes_raw = multiple_results['boxes'].tolist()
print(f"{len(multiple_results['boxes'])} objects")

In [ ]:
# visualize_detections(image, multiple_results)

In [ ]:
masks = segment_objects(models['sam_model'], models['sam_processor'], image, boxes_raw)
print(f"{len(masks)} masks")

In [ ]:
annotations = analyze_objects_with_bioclip(
    models, image, result, masks, species_candidates
)

print(f"{len(annotations)} analyzed")

In [ ]:
final_output = {
    "img_path": IMAGE_PATH,
    "annotations": annotations
}

# print(json.dumps(final_output, indent=2, ensure_ascii=False))

In [ ]:
def visualize_detections(image: Image.Image, detection_results: Dict[str, Any]):
    image_to_draw = image.copy()
    draw = ImageDraw.Draw(image_to_draw)

    font = ImageFont.load_default()

    labels = list(set(detection_results["text_labels"]))
    color_palette = {label: (random.randint(100, 255), random.randint(0, 150), random.randint(0, 150)) for label in labels}

    for box, score, label in zip(detection_results['boxes'], detection_results['scores'], detection_results['text_labels']):
        box = box.tolist()
        xmin, ymin, xmax, ymax = map(int, box)
        color = color_palette.get(label, "red")

        draw.rectangle([xmin, ymin, xmax, ymax], outline=color, width=4)

        text = f"{label}: {score:.2f}"

        text_bbox = draw.textbbox((xmin, ymin), text, font=font)

        text_y_position = ymin - (text_bbox[3] - text_bbox[1]) - 5
        if text_y_position < 0:
            text_y_position = ymin + 5

        text_bbox_moved = (xmin, text_y_position, xmin + (text_bbox[2] - text_bbox[0]), text_y_position + (text_bbox[3] - text_bbox[1]))

        draw.rectangle(text_bbox_moved, fill=color)
        draw.text((xmin, text_y_position), text, fill="white", font=font)

    plt.figure(figsize=(8, 8))
    plt.imshow(image_to_draw)
    plt.axis("off")
    plt.show()

In [ ]:
def visualize_results(
    image: Image.Image,
    analysis_data: Dict,
    masks: List[np.ndarray],
):
    plt.figure(figsize=(8, 8))
    ax = plt.gca()
    ax.imshow(image)
    ax.axis('off')

    for i, ann in enumerate(analysis_data.get("annotations", [])):
        if i < len(masks):
            mask = masks[i]
            if mask.ndim == 2 and np.any(mask):
                color = np.random.rand(3)
                h, w = mask.shape
                mask_overlay = np.zeros((h, w, 4), dtype=np.float32)
                mask_overlay[mask, :3] = color
                mask_overlay[mask, 3] = 0.4
                ax.imshow(mask_overlay)

        x1, y1, x2, y2 = ann['bbox']
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2.5, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)

        if ann['species_probabilities']:
            top_species = ann['species_probabilities'][0]
            text = f"#{i+1}: {top_species['species']} ({top_species['probability']:.2f})"
        else:
            text = f"#{i+1}: Вид не определен"

        ax.text(x1, y1 - 10, text, color='white', fontsize=12,
                bbox=dict(facecolor='black', alpha=0.7, pad=2))

    plt.tight_layout()
    plt.show()

In [ ]:
final_output

In [ ]:
visualize_results(image, final_output, masks)

In [ ]:
image_path = "hole.jpg"

image = Image.open(image_path).convert("RGB")

In [ ]:
plt.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
def find_hollows_on_trunk(
    dino_model,
    dino_processor,
    full_image: Image.Image,
    trunk_box: List[int],
    max_hollow_to_trunk_ratio: float = 0.5,
) -> List[Dict[str, Any]]:
    if not trunk_box or len(trunk_box) != 4:
        return []

    trunk_crop = full_image.crop(tuple(trunk_box))

    hollow_prompts = [["a dark hole in the wood", "a tree cavity",
                       "an opening in the trunk", "a black spot on bark"]]

    inputs = dino_processor(images=trunk_crop, text=hollow_prompts, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = dino_model(**inputs)

    results = dino_processor.post_process_grounded_object_detection(
        outputs,
        threshold=0.4,
        text_threshold=0.25,
        target_sizes=[trunk_crop.size[::-1]]
    )[0]

    if len(results['boxes']) == 0:
        return []

    found_hollows = []
    if len(results['boxes']) > 0:
        for box, score in zip(results['boxes'], results['scores']):
            local_box = [int(i) for i in box.tolist()]
            global_box = [
                local_box[0] + trunk_box[0],
                local_box[1] + trunk_box[1],
                local_box[2] + trunk_box[0],
                local_box[3] + trunk_box[1],
            ]
            found_hollows.append({"box": global_box, "score": score.item()})

    return found_hollows

    # plausible_hollows = []
    # trunk_area = (trunk_box[2] - trunk_box[0]) * (trunk_box[3] - trunk_box[1])

    # for box, score in zip(results['boxes'], results['scores']):
    #     local_box = [int(i) for i in box.tolist()]

    #     hollow_area = (local_box[2] - local_box[0]) * (local_box[3] - local_box[1])
    #     if hollow_area / trunk_area > max_hollow_to_trunk_ratio:
    #         continue

    #     global_box = [
    #         local_box[0] + trunk_box[0],
    #         local_box[1] + trunk_box[1],
    #         local_box[2] + trunk_box[0],
    #         local_box[3] + trunk_box[1],
    #     ]
    #     plausible_hollows.append({
    #         "box": global_box,
    #         "score": score.item(),
    #         "area": hollow_area
    #     })

    # if not plausible_hollows:
    #     return []

    # smallest_hollow = min(plausible_hollows, key=lambda x: x['area'])

    # del smallest_hollow['area']

    # return [smallest_hollow]

In [ ]:
def visualize_hollow_detection(
    image: Image.Image,
    trunk_box: List[int],
    trunk_score: float,
    hollows_data: List[Dict[str, Any]]
):
    img_to_draw = image.copy()
    draw = ImageDraw.Draw(img_to_draw)

    font = ImageFont.load_default()

    draw.rectangle(tuple(trunk_box), outline="blue", width=4)
    trunk_label = f"Ствол: {trunk_score:.2f}"
    draw.text((trunk_box[0], trunk_box[1] - 25), trunk_label, fill="blue", font=font)

    if hollows_data:
        for hollow in hollows_data:
            box = hollow['box']
            score = hollow['score']
            draw.rectangle(tuple(box), outline="red", width=5)
            hollow_label = f"hole: {score:.2f}"
            draw.text((box[0], box[1] - 10), hollow_label, fill="red", font=font)

    plt.figure(figsize=(12, 12))
    plt.imshow(img_to_draw)
    plt.title("Результат поиска дупла")
    plt.axis("off")
    plt.show()

In [ ]:
trunk_prompts = [["tree trunk", "bark"]]
inputs = models['dino_processor'](images=image, text=trunk_prompts, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    outputs = models['dino_model'](**inputs)
results = models['dino_processor'].post_process_grounded_object_detection(
    outputs, threshold=0.4, text_threshold=0.25, target_sizes=[image.size[::-1]]
)[0]

In [ ]:
if len(results['boxes']) > 0:
    main_trunk_box = [int(i) for i in results['boxes'][0].tolist()]
    main_trunk_score = results['scores'][0].item()

    found_hollows = find_hollows_on_trunk(
        models['dino_model'],
        models['dino_processor'],
        image,
        main_trunk_box,
    )

    visualize_hollow_detection(image, main_trunk_box, main_trunk_score, found_hollows)

    has_hollow = len(found_hollows) > 0

    print(has_hollow)

In [ ]:
# dinov3_transform = T.Compose([
#     T.ToTensor(),
#     T.Resize((560, 560), antialias=True), # Размер, кратный PATCH_SIZE
#     T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
# ])

In [ ]:
# device = "cuda" if torch.cuda.is_available() else "cpu"
# processor = AutoImageProcessor.from_pretrained("facebook/dinov3-vits16-pretrain-lvd1689m")
# model = AutoModel.from_pretrained("facebook/dinov3-vits16-pretrain-lvd1689m").to(device)
# patch_size = model.config.patch_size
# num_register_tokens = model.config.num_register_tokens